### All MySQL commands using Python for automating MySQL

In [1]:
import mysql.connector

DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def run(sql, description=""):
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute(sql)
        conn.commit()
        print(f"✅ {description or 'OK'} | rows affected: {cur.rowcount}")
    except Exception as e:
        print(f"❌ {description}: {e}")
    finally:
        cur.close(); conn.close()

def fetch(sql):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    cur.close(); conn.close()
    return cols, rows

In [2]:
run("""
CREATE TABLE IF NOT EXISTS dim_neo (
    neo_id VARCHAR(20) PRIMARY KEY,
    full_name VARCHAR(150),
    is_hazardous BOOLEAN,
    diameter_km_avg FLOAT,
    absolute_magnitude_h FLOAT,
    eccentricity FLOAT,
    semi_major_axis_au FLOAT,
    inclination_deg FLOAT,
    orbital_period_days FLOAT,
    data_arc_days INT,
    n_observations INT,
    impact_probability FLOAT,
    palermo_scale_max FLOAT,
    torino_scale INT,
    last_obs_date VARCHAR(20)
)
""", "create dim_neo")

run("""
CREATE TABLE IF NOT EXISTS fact_close_approach (
    approach_id INT AUTO_INCREMENT PRIMARY KEY,
    neo_id VARCHAR(20),
    close_approach_date DATE,
    relative_velocity_kmh FLOAT,
    miss_distance_km FLOAT,
    miss_distance_ld FLOAT,
    orbiting_body VARCHAR(50),
    FOREIGN KEY (neo_id) REFERENCES dim_neo(neo_id)
)
""", "create fact_close_approach")

✅ create dim_neo | rows affected: 0
✅ create fact_close_approach | rows affected: 0


In [5]:
run("""
INSERT INTO dim_neo (neo_id, full_name, diameter_km_avg, absolute_magnitude_h,
    eccentricity, semi_major_axis_au, inclination_deg, orbital_period_days,
    data_arc_days, n_observations, impact_probability, palermo_scale_max,
    torino_scale, last_obs_date, is_hazardous)
SELECT
    o.neo_id, TRIM(o.full_name),
    AVG((f.est_diameter_min_km + f.est_diameter_max_km) / 2),
    NULL, o.eccentricity, o.semi_major_axis_au, o.inclination_deg,
    o.orbital_period_days, o.data_arc_days, o.n_observations,
    s.impact_probability, s.palermo_scale_max, s.torino_scale,
    s.last_obs_date, MAX(f.is_hazardous)
FROM raw_orbital_elements o
LEFT JOIN raw_sentry_risk s ON o.neo_id = s.neo_id
LEFT JOIN raw_neo_feed f ON o.neo_id = f.neo_id
GROUP BY o.neo_id, o.full_name, o.eccentricity, o.semi_major_axis_au,
         o.inclination_deg, o.orbital_period_days, o.data_arc_days,
         o.n_observations, s.impact_probability, s.palermo_scale_max,
         s.torino_scale, s.last_obs_date
""", "populate dim_neo")

✅ populate dim_neo | rows affected: 2000


In [7]:
run("""
INSERT INTO fact_close_approach (neo_id, close_approach_date, relative_velocity_kmh,
    miss_distance_km, miss_distance_ld, orbiting_body)
SELECT f.neo_id, f.close_approach_date, f.relative_velocity_kmh, f.miss_distance_km,
    f.miss_distance_km / 384400, f.orbiting_body
FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.neo_id = d.neo_id
""", "populate fact_close_approach")

✅ populate fact_close_approach | rows affected: 0


In [8]:
cols, rows = fetch("SELECT COUNT(*) FROM raw_neo_feed")
print("raw_neo_feed total:", rows[0][0])
cols, rows = fetch("SELECT COUNT(DISTINCT neo_id) FROM raw_neo_feed")
print("raw_neo_feed unique neo_id:", rows[0][0])
cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach inserted:", rows[0][0])


raw_neo_feed total: 40
raw_neo_feed unique neo_id: 40
fact_close_approach inserted: 0


In [10]:
import pandas as pd

In [11]:
cols, rows = fetch("SELECT neo_id, name FROM raw_neo_feed LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

cols, rows = fetch("SELECT neo_id, full_name FROM dim_neo LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

    neo_id                name
0  2240320  240320 (2003 HS42)
1  2250680   250680 (2005 QC5)
2  2452639   452639 (2005 UY6)
3  2500136  500136 (2012 CO46)
4  2523808  523808 (2007 ML24)
     neo_id               full_name
0  20000433      433 Eros (A898 PA)
1  20000719    719 Albert (A911 TB)
2  20000887    887 Alinda (A918 AA)
3  20001036  1036 Ganymed (A924 UB)
4  20001221    1221 Amor (1932 EA1)


In [12]:
run("ALTER TABLE raw_neo_feed ADD COLUMN designation_num VARCHAR(20)", "add designation_num to raw_neo_feed")
run("ALTER TABLE dim_neo ADD COLUMN designation_num VARCHAR(20)", "add designation_num to dim_neo")

run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
""", "populate designation_num in raw_neo_feed")

run("""
UPDATE dim_neo
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(full_name), ' ', 1))
""", "populate designation_num in dim_neo")

✅ add designation_num to raw_neo_feed | rows affected: 0
✅ add designation_num to dim_neo | rows affected: 0
✅ populate designation_num in raw_neo_feed | rows affected: 40
✅ populate designation_num in dim_neo | rows affected: 2000


In [13]:
cols, rows = fetch("""
SELECT COUNT(*) FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""")
print("matched rows:", rows[0][0])

matched rows: 2


In [20]:
import requests
import mysql.connector

API_KEY = "hBnrFPk07qCcnQy1unyYO2VDgD1nAKH9gi8liBHu"
DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def fetch_neo_feed(start_date, end_date):
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["near_earth_objects"]

def insert_feed_data(data):
    conn = get_conn()
    cur = conn.cursor()
    for date, objects in data.items():
        for obj in objects:
            approach = obj["close_approach_data"][0]
            cur.execute("""
                INSERT IGNORE INTO raw_neo_feed
                (neo_id, name, absolute_magnitude_h, est_diameter_min_km, est_diameter_max_km,
                 is_hazardous, close_approach_date, relative_velocity_kmh, miss_distance_km, orbiting_body)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """, (
                obj["id"], obj["name"], obj["absolute_magnitude_h"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_min"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
                obj["is_potentially_hazardous_asteroid"], date,
                float(approach["relative_velocity"]["kilometers_per_hour"]),
                float(approach["miss_distance"]["kilometers"]),
                approach["orbiting_body"]
            ))
    conn.commit()
    cur.close(); conn.close()

In [21]:
import time
from datetime import datetime, timedelta

today = datetime.today()
for i in range(0, 180, 7):  # ~6 months, week by week
    start = (today - timedelta(days=i+7)).strftime("%Y-%m-%d")
    end = (today - timedelta(days=i)).strftime("%Y-%m-%d")
    try:
        insert_feed_data(fetch_neo_feed(start, end))
        print(f"✅ {start} to {end}")
    except Exception as e:
        print(f"❌ {start} to {end}: {e}")
    time.sleep(1)

✅ 2026-07-04 to 2026-07-11
✅ 2026-06-27 to 2026-07-04
✅ 2026-06-20 to 2026-06-27
✅ 2026-06-13 to 2026-06-20
✅ 2026-06-06 to 2026-06-13
✅ 2026-05-30 to 2026-06-06
✅ 2026-05-23 to 2026-05-30
✅ 2026-05-16 to 2026-05-23
✅ 2026-05-09 to 2026-05-16
✅ 2026-05-02 to 2026-05-09
✅ 2026-04-25 to 2026-05-02
✅ 2026-04-18 to 2026-04-25
✅ 2026-04-11 to 2026-04-18
✅ 2026-04-04 to 2026-04-11
✅ 2026-03-28 to 2026-04-04
✅ 2026-03-21 to 2026-03-28
✅ 2026-03-14 to 2026-03-21
✅ 2026-03-07 to 2026-03-14
✅ 2026-02-28 to 2026-03-07
✅ 2026-02-21 to 2026-02-28
✅ 2026-02-14 to 2026-02-21
✅ 2026-02-07 to 2026-02-14
✅ 2026-01-31 to 2026-02-07
✅ 2026-01-24 to 2026-01-31
✅ 2026-01-17 to 2026-01-24
✅ 2026-01-10 to 2026-01-17


In [22]:
import requests

def fetch_orbital_by_designation(designation):
    url = "https://ssd-api.jpl.nasa.gov/sbdb.api"
    params = {"sstr": designation, "full-prec": "true"}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return None
    return r.json()

# get distinct designations from your (now much bigger) feed table
cols, rows = fetch("SELECT DISTINCT designation_num FROM raw_neo_feed")
designations = [r[0] for r in rows]
print(f"{len(designations)} unique objects to fetch")

20 unique objects to fetch


In [23]:
run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
WHERE designation_num IS NULL
""", "backfill designation_num for new rows")

✅ backfill designation_num for new rows | rows affected: 819


In [24]:
cols, rows = fetch("SELECT COUNT(*) FROM raw_neo_feed")
print("total feed rows:", rows[0][0])

cols, rows = fetch("SELECT DISTINCT designation_num FROM raw_neo_feed")
designations = [r[0] for r in rows]
print(f"{len(designations)} unique objects to fetch")

total feed rows: 859
118 unique objects to fetch


In [25]:
import requests
import time

def fetch_orbital_by_designation(designation):
    url = "https://ssd-api.jpl.nasa.gov/sbdb.api"
    params = {"sstr": designation, "full-prec": "true"}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return None
    return r.json()

results = {}
for i, des in enumerate(designations):
    try:
        data = fetch_orbital_by_designation(des)
        if data and "orbit" in data:
            results[des] = data
            print(f"✅ {i+1}/{len(designations)} {des}")
        else:
            print(f"⚠️ {i+1}/{len(designations)} {des} — no orbit data")
    except Exception as e:
        print(f"❌ {des}: {e}")
    time.sleep(0.5)  # respect rate limit

✅ 1/118 1943
✅ 2/118 2340
✅ 3/118 17182
✅ 4/118 52340
✅ 5/118 54071
✅ 6/118 88710
✅ 7/118 90416
✅ 8/118 136818
✅ 9/118 141495
✅ 10/118 141531
✅ 11/118 152637
✅ 12/118 162882
✅ 13/118 192559
✅ 14/118 240320
✅ 15/118 242708
✅ 16/118 244670
✅ 17/118 248590
✅ 18/118 250680
✅ 19/118 276033
✅ 20/118 276891
✅ 21/118 302831
✅ 22/118 310842
✅ 23/118 318411
✅ 24/118 326290
✅ 25/118 330659
✅ 26/118 363599
✅ 27/118 367943
✅ 28/118 374038
✅ 29/118 375103
✅ 30/118 376879
✅ 31/118 388945
✅ 32/118 394392
✅ 33/118 398188
✅ 34/118 399325
✅ 35/118 427684
✅ 36/118 434196
✅ 37/118 437844
✅ 38/118 439877
✅ 39/118 441987
✅ 40/118 452639
✅ 41/118 454101
✅ 42/118 467336
✅ 43/118 469219
✅ 44/118 470310
✅ 45/118 476187
✅ 46/118 478784
✅ 47/118 480858
✅ 48/118 488615
✅ 49/118 494975
✅ 50/118 497626
✅ 51/118 499998
✅ 52/118 500080
✅ 53/118 500136
✅ 54/118 509352
✅ 55/118 510190
✅ 56/118 513529
✅ 57/118 515049
✅ 58/118 523808
✅ 59/118 526798
✅ 60/118 527977
✅ 61/118 530520
✅ 62/118 530974
✅ 63/118 537395
✅ 64/118 6

In [26]:
import re

def clean_designation(name):
    name = name.strip()
    if name.startswith("("):
        # provisional designation, e.g. "(2020 AB1)" -> "2020 AB1"
        return name.strip("()")
    else:
        # numbered asteroid, e.g. "240320 (2003 HS42)" -> "240320"
        return name.split(" ")[0]

# rebuild designations list correctly from raw_neo_feed
cols, rows = fetch("SELECT DISTINCT name FROM raw_neo_feed")
designations = [clean_designation(r[0]) for r in rows]
print(f"{len(designations)} unique objects to fetch")
print(designations[:10])

836 unique objects to fetch
['1943', '2340', '17182', '52340', '54071', '88710', '90416', '136818', '141495', '141531']


In [27]:
run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(TRAILING ')' FROM TRIM(LEADING '(' FROM TRIM(name)))
WHERE name LIKE '(%'
""", "fix provisional designation_num")

run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
WHERE name NOT LIKE '(%'
""", "fix numbered designation_num")

✅ fix provisional designation_num | rows affected: 767
✅ fix numbered designation_num | rows affected: 0


In [28]:
cols, rows = fetch("SELECT DISTINCT name FROM raw_neo_feed")
designations = [clean_designation(r[0]) for r in rows]
print(f"{len(designations)} unique objects to fetch")

836 unique objects to fetch


In [29]:
results = {}
for i, des in enumerate(designations):
    try:
        data = fetch_orbital_by_designation(des)
        if data and "orbit" in data:
            results[des] = data
            print(f"✅ {i+1}/{len(designations)} {des}")
        else:
            print(f"⚠️ {i+1}/{len(designations)} {des} — no orbit data")
    except Exception as e:
        print(f"❌ {des}: {e}")
    time.sleep(0.5)

✅ 1/836 1943
✅ 2/836 2340
✅ 3/836 17182
✅ 4/836 52340
✅ 5/836 54071
✅ 6/836 88710
✅ 7/836 90416
✅ 8/836 136818
✅ 9/836 141495
✅ 10/836 141531
✅ 11/836 152637
✅ 12/836 162882
✅ 13/836 192559
✅ 14/836 240320
✅ 15/836 242708
✅ 16/836 244670
✅ 17/836 248590
✅ 18/836 250680
✅ 19/836 276033
✅ 20/836 276891
✅ 21/836 302831
✅ 22/836 310842
✅ 23/836 318411
✅ 24/836 326290
✅ 25/836 330659
✅ 26/836 363599
✅ 27/836 367943
✅ 28/836 374038
✅ 29/836 375103
✅ 30/836 376879
✅ 31/836 388945
✅ 32/836 394392
✅ 33/836 398188
✅ 34/836 399325
✅ 35/836 427684
✅ 36/836 434196
✅ 37/836 437844
✅ 38/836 439877
✅ 39/836 441987
✅ 40/836 452639
✅ 41/836 454101
✅ 42/836 467336
✅ 43/836 469219
✅ 44/836 470310
✅ 45/836 476187
✅ 46/836 478784
✅ 47/836 480858
✅ 48/836 488615
✅ 49/836 494975
✅ 50/836 497626
✅ 51/836 499998
✅ 52/836 500080
✅ 53/836 500136
✅ 54/836 509352
✅ 55/836 510190
✅ 56/836 513529
✅ 57/836 515049
✅ 58/836 523808
✅ 59/836 526798
✅ 60/836 527977
✅ 61/836 530520
✅ 62/836 530974
✅ 63/836 537395
✅ 64/836 6

In [31]:
def insert_dim_neo_from_sbdb(results):
    conn = get_conn()
    cur = conn.cursor()
    for des, data in results.items():
        obj = data.get("object", {})
        orbit = data.get("orbit", {})
        elems = {e["name"]: e["value"] for e in orbit.get("elements", [])}
        spkid = obj.get("spkid")
        full_name = obj.get("fullname")

        cur.execute("""
            INSERT INTO dim_neo
            (neo_id, full_name, designation_num, eccentricity, semi_major_axis_au,
             inclination_deg, orbital_period_days, data_arc_days, n_observations)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON DUPLICATE KEY UPDATE full_name=VALUES(full_name)
        """, (
            spkid, full_name, des,
            float(elems.get("e", 0) or 0),
            float(elems.get("a", 0) or 0),
            float(elems.get("i", 0) or 0),
            float(elems.get("per", 0) or 0),
            None, None
        ))
    conn.commit()
    cur.close(); conn.close()

insert_dim_neo_from_sbdb(results)

In [32]:
insert_dim_neo_from_sbdb(results)

In [33]:
cols, rows = fetch("SELECT COUNT(*) FROM dim_neo")
print("dim_neo rows:", rows[0][0])

cols, rows = fetch("""
SELECT COUNT(*) FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""")
print("matched rows for fact table:", rows[0][0])

dim_neo rows: 2760
matched rows for fact table: 821


In [34]:
run("ALTER TABLE dim_neo ADD COLUMN priority_score FLOAT", "add priority_score column")

run("""
UPDATE dim_neo
SET priority_score = 
    (COALESCE(impact_probability,0) * 100000) +
    (CASE WHEN is_hazardous = 1 THEN 20 ELSE 0 END) +
    (CASE WHEN data_arc_days < 365 THEN 15 ELSE 0 END) +
    (CASE WHEN n_observations < 50 THEN 15 ELSE 0 END) +
    (CASE WHEN torino_scale > 0 THEN 30 ELSE 0 END)
""", "compute priority_score")

✅ add priority_score column | rows affected: 0
✅ compute priority_score | rows affected: 2760


In [35]:
cols, rows = fetch("SELECT COUNT(*) FROM dim_neo")
print("dim_neo rows:", rows[0][0])

cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach rows:", rows[0][0])

cols, rows = fetch("SELECT full_name, priority_score FROM dim_neo ORDER BY priority_score DESC LIMIT 10")
import pandas as pd
print(pd.DataFrame(rows, columns=cols))

dim_neo rows: 2760
fact_close_approach rows: 0
                   full_name  priority_score
0        447977 (2008 CC119)            15.0
1         434053 (2001 UP27)            15.0
2  367943 Duende (2012 DA14)            15.0
3         163015 (2001 UX16)            15.0
4       719 Albert (A911 TB)             0.0
5       887 Alinda (A918 AA)             0.0
6       (2026 EU3 = 2026 FM)             0.0
7       1221 Amor (1932 EA1)             0.0
8      1566 Icarus (1949 MA)             0.0
9  1620 Geographos (1951 RA)             0.0


In [36]:
run("""
INSERT INTO fact_close_approach (neo_id, close_approach_date, relative_velocity_kmh,
    miss_distance_km, miss_distance_ld, orbiting_body)
SELECT d.neo_id, f.close_approach_date, f.relative_velocity_kmh, f.miss_distance_km,
    f.miss_distance_km / 384400, f.orbiting_body
FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""", "populate fact_close_approach")

✅ populate fact_close_approach | rows affected: 821


In [37]:
cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach rows:", rows[0][0])

fact_close_approach rows: 821


In [38]:
run("""
UPDATE dim_neo d
INNER JOIN raw_neo_feed f ON d.designation_num = f.designation_num
SET d.is_hazardous = f.is_hazardous
WHERE d.is_hazardous IS NULL
""", "backfill is_hazardous")

✅ backfill is_hazardous | rows affected: 799


In [39]:
run("""
UPDATE dim_neo
SET priority_score = 
    (COALESCE(impact_probability,0) * 100000) +
    (CASE WHEN is_hazardous = 1 THEN 20 ELSE 0 END) +
    (CASE WHEN data_arc_days < 365 THEN 15 ELSE 0 END) +
    (CASE WHEN n_observations < 50 THEN 15 ELSE 0 END) +
    (CASE WHEN torino_scale > 0 THEN 30 ELSE 0 END)
""", "recompute priority_score")

✅ recompute priority_score | rows affected: 78


In [40]:
cols, rows = fetch("SELECT full_name, priority_score FROM dim_neo ORDER BY priority_score DESC LIMIT 10")
print(pd.DataFrame(rows, columns=cols))

               full_name  priority_score
0      152637 (1997 NC1)            20.0
1    276033 (2002 AJ129)            20.0
2       192559 (1998 VO)            20.0
3     162882 (2001 FD58)            20.0
4       302831 (2003 FH)            20.0
5      250680 (2005 QC5)            20.0
6     141495 (2002 EZ11)            20.0
7      242708 (2005 UK1)            20.0
8     90416 (2003 YK118)            20.0
9  2340 Hathor (1976 UA)            20.0
